In [187]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from pathlib import Path
import time
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
from glob import glob

In [252]:
folder = Path("html_pages/")
html_files = list(folder.glob("*.html"))

In [271]:
file = np.random.choice(html_files)

recipes = []
for file in html_files:
    try:
        with file.open(encoding="utf-8", errors="ignore") as f:
            soup = BeautifulSoup(f, "html.parser")
        body = soup.html.body
        centered = body.find("div", class_="centered")
        for row in centered.find_all("div", class_="row"):
            if row.get("class") == ["row"]:
                for column in row.find_all("div", class_="column"):
                    if column.get("class") == ["column", "grid_10c"]:
                        break
        table = column.find("table", border="0", cellpadding="0", cellspacing="0", width="100%").tbody
        recipe_name = table.find_all("tr")[1].text[:-1]
        recipe_ingr = table.find("div", style="padding-left: 1rem; color: BLACK;")
        recipe_ingr = recipe_ingr.get_text(separator="\n").splitlines()
        recipe_ingr = [line.strip() for line in recipe_ingr if line.strip()]
        recipe_inst = table.find("div", style="color: #772222;")

        recipes.append([recipe_name, recipe_ingr])
    except:
        print(f'Failed to fetch {file}')

with open('data/cookscom_recipes.pkl', 'wb') as f:
    pickle.dump(recipes, f)

print('-----------------')
print("Sample recipe: ")
print(recipe_name)
print('-----------------')
[print(line) for line in recipe_ingr]
print('-----------------')
print(recipe_inst)

Failed to fetch html_pages/Italian Egg Patties - Recipe - Cooks.com.html
Failed to fetch html_pages/Patty Melt Sandwich - Recipe - Cooks.com.html
Failed to fetch html_pages/Mushroom Burgers 3 - Recipe - Cooks.com.html
Failed to fetch html_pages/Easy Sauce for Sirloin Patties - Recipe - Cooks.com.html
Failed to fetch html_pages/Hamburger Patties 4 - Recipe - Cooks.com.html
Failed to fetch html_pages/Ham Patties with Sweet Potatoes, Pineapple & Bacon - Recipe - Cooks.com.html
Failed to fetch html_pages/Summer Squash Creamed Patty Pan - Recipe - Cooks.com.html
Failed to fetch html_pages/Salmon Patties 70 - Recipe - Cooks.com.html
Failed to fetch html_pages/Chili Patty Melt Burger - Recipe - Cooks.com.html
Failed to fetch html_pages/Hot Dog, Chili Burger Sauce - Recipe - Cooks.com.html
Failed to fetch html_pages/Patti's Yams - Recipe - Cooks.com.html
Failed to fetch html_pages/Bbq Beef Burgers - Recipe - Cooks.com.html
Failed to fetch html_pages/Bacon - Cheese, Potato Patties - Recipe - Co

In [243]:
# filter non-animal recipes
animal_prods = ['beef', 'chuck', 'lamb', 'hamburger', 'chicken', 'turkey', 'fish', 
                'salmon', 'tuna', 'pork', 'bacon', 'buffalo', 'meat', 'oyster', 
                'deer', 'ground round', 'venison', 'sirloin', 'mackarel', 'mackerel', 'spam',
                'ham', 'shrimp', ' cod ', 'burger patt', 'steak', 'bologna', 'clam',
                'veal', 'filet', 'crab', 'carp', ]

# Remove all recipes that contain animal products
include_idx = np.ones(len(recipes))
for i, recipe in enumerate(recipes):
    for animal_prod in animal_prods:
        if animal_prod in str(recipe[1]).lower():
            include_idx[i] = 0

irrelevant = ['mint', 'soup', 'bun', 'sauce', 'relish', 'peanut', 'chocolate', 'cookie', 
              'patti\'s', 'fudge', 'patty\'s', 'salmon', 'chicken', 'beef']
# Remove irrelevant products
for i, recipe in enumerate(recipes):
    for item in irrelevant:
        if item in str(recipe[0]).lower():
            include_idx[i] = 0

veg_recipes = []
for i in range(len(include_idx)):
    if include_idx[i] == 1:
        veg_recipes.append(recipes[i])

print(len(veg_recipes))

300


In [244]:
# Create a word cloud of recipe names
allnames = []
for recipe in veg_recipes:
    name = recipe[0].lower().strip().split(' ')
    allnames.extend(name)
allnames = np.array(allnames)

# excluded words
excluded = ['patty','patties', 'burger', 'burgers', 'fried', 'and', '-', 'pan', 'italian', 
            'black', 'favorite', 'fresh', 'grilled', 'mashed', 'loaf']
mask = np.ones(len(allnames), dtype=bool)
for i in range(len(mask)):
    if allnames[i] in excluded:
        mask[i] = 0
allnames = allnames[mask]

words, counts = np.unique(allnames, return_counts=True)
idx_sort = np.argsort(counts)[::-1]
words, counts = words[idx_sort], counts[idx_sort]

for i in range(20):
    print(counts[i], words[i])

31 potato
26 squash
24 zucchini
20 bean
16 oatmeal
15 cheese
11 tofu
10 cottage
9 lentil
8 eggplant
8 vegetarian
8 rice
8 mushroom
7 okra
6 pecan
6 nut
5 sweet
5 oat
5 vegetable
5 veggie
